# OpenHeadMouse: Colab session viewer
Run the setup, loading and session-viewer cells in order. Loading controls and the viewer have separate outputs, sharing the same imported sessions.
Use a CPU runtime; no camera, mouse driver, desktop GUI or GPU is needed.

**Privacy:** recordings uploaded here go to your Google Colab runtime, not to GitHub or any playback service. 
Do not share a notebook with saved outputs containing participant data. Runtime files and imports are temporary; download the summary CSV before disconnecting.

**Setup:** the repository/branch below must contain `colab_session_viewer`. 
If working with unpublished changes, upload the complete project folder to the runtime first and set `PROJECT_DIR` to it.

In [1]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/simonjudge/OpenHeadMouse.git'
REVISION = ''  # Optional published branch or tag; empty uses the default branch.
# Prefer this notebook's local project over an older nested clone.
PROJECT_DIR = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                    if (folder / 'colab_session_viewer' / 'app.py').is_file()
                    and (folder / 'core').is_dir()), Path.cwd() / 'OpenHeadMouse')

if not PROJECT_DIR.exists():
    command = ['git', 'clone', '--depth', '1']
    if REVISION:
        command += ['--branch', REVISION]
    subprocess.check_call(command + [REPOSITORY_URL, str(PROJECT_DIR)])
requirements = PROJECT_DIR / 'colab_session_viewer' / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError('This checkout does not contain the Colab viewer. Select its published branch or upload the complete changed project.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
PROJECT_DIR = PROJECT_DIR.resolve()
sys.path = [str(PROJECT_DIR)] + [entry for entry in sys.path if entry != str(PROJECT_DIR)]
loaded_viewer = sys.modules.get('colab_session_viewer')
if loaded_viewer is not None and Path(loaded_viewer.__file__).resolve().parent != PROJECT_DIR / 'colab_session_viewer':
    raise RuntimeError('Another viewer checkout is already loaded. Restart the kernel and rerun setup.')

from session_viewer.mesh_visualizer import download_mediapipe_model
download_mediapipe_model()
print('Ready. Run the loading cell, then the session-viewer cell below.')


Ready. Run the loading cell, then the session-viewer cell below.


In [2]:
try:
    from google.colab import output
except ModuleNotFoundError as error:
    if error.name not in ('google', 'google.colab'):
        raise
    print('Local notebook runtime; Colab widget manager not needed.')
else:
    output.enable_custom_widget_manager()
import importlib
import colab_session_viewer.data as viewer_data
import colab_session_viewer.archive as viewer_archive
import colab_session_viewer.app as viewer_app
for module in (viewer_data, viewer_archive, viewer_app):
    importlib.reload(module)
from colab_session_viewer.app import NotebookViewer
print('Viewer loaded from:', viewer_app.__file__)

DEMO_DATA_DIR = str(PROJECT_DIR / 'colab_session_viewer' / 'demodata')  # Bundled demo ZIPs; works locally and in Colab.
DEMO_MODE = False  # True: automatically load demos and show only 'Loaded Demo Data' instead of import controls.
if 'viewer' not in globals():
    viewer = NotebookViewer(demo_directory=DEMO_DATA_DIR, demo_mode=DEMO_MODE)
viewer.show_loading();


Local notebook runtime; Colab widget manager not needed.
Viewer loaded from: C:\Users\judges.BDGH\Documents\VSCode\HeadMouse\colab_session_viewer\app.py


## Session viewer and participant summaries
Run this cell after the loading cell. Imports above update this output automatically. Rerunning this cell does not clear your loaded sessions.


In [3]:
viewer.show_session();


## Importing multiple participants and sessions
Loading controls and the session viewer are separate cells sharing one collection. Run both once; later imports update the viewer automatically. Restart the kernel to load code changes for an existing viewer.

**Demo data:** check **Load Demo Data** to import ZIPs bundled in `colab_session_viewer/demodata`. This works locally and in Colab without Google Drive authorization. Each ZIP needs a valid root CSV manifest. You can override `DEMO_DATA_DIR` or edit **Demo folder** to use another local folder. The full new batch is validated before import. Unchecking keeps loaded data; rechecking skips loaded ZIP paths and imports new ones. Errors reset the checkbox so you can fix the folder/files and retry. Restart the kernel to reload changed ZIPs at the same path.

For each test, enter a participant (for example `P01`) and session (`Test 1`) and choose a head JSON/CSV, a Fitts JSON, or both. 
Click **Add session**, then repeat with `P01 / Test 2`, `P02 / Test 1`, etc. Choose both files only when recorded together with overlapping host timestamps. 
Legacy files without host timestamps can be imported separately; they are never silently aligned.

Upload a ZIP **with or without a manifest** using **Dataset ZIP**, then **Import dataset ZIP**. Without a manifest, the viewer detects recordings by their contents and suggests pairs from overlapping shared host timestamps. Review file paths, timestamps, sample/target counts and Fitts notes alongside the editable participant/session labels. The Participant input above the uploads prefills every review row; labels remain editable. Ambiguous matches and legacy streams remain separate; participants are not guessed from timestamps.

Edit the head/Fitts dropdowns to correct pairings, use **Add assignment row** to split pairs, or untick unwanted rows. Each file can be included once per import. Click **Import labelled sessions**, then **Download manifest.csv for ZIP import N** to save the assignments. Put the downloaded manifest at the root of the original ZIP to reuse those labels and pairings. Matching streams stay on their shared host clock, not an artificial start-time alignment.

Any root-level CSV filename (for example `A manifest.csv`) is checked for a valid manifest. A valid manifest imports only its listed files, not other recordings in the ZIP. Multiple valid manifests or malformed assignments produce an explicit error. Unrelated CSVs are not treated as manifests. Required columns:
```csv
participant,session,head_file,fitts_file,selection_method,dwell_time_s
P01,Test 1,P01/head1.json,P01/fitts1.json,external_switch,
P01,Test 2,P01/head2.csv,P01/fitts2.json,external_switch,
P02,Test 1,,P02/fitts1.json,dwell,0.6
```
Paths are relative to the ZIP, use `/` separators, and may be blank when that stream was not recorded. 
Optional selection labels are `external_switch`, `dwell`, `mouse`, `other` (non-dwell), or `unknown`. Dwell requires a positive `dwell_time_s` in seconds; leave duration blank for other methods. Old four-column manifests remain valid with unknown selection. Individual-file imports and ZIP review rows have the same controls. No method is inferred from participant identity.
You can import multiple ZIPs without clearing earlier data; each has a separate downloadable manifest. Duplicate participant/session names are rejected rather than overwritten. Manifests store assignments, not source data or exclusion settings. Keep the original ZIP as well.

**Exclusions** is at the top of the viewer, before playback. A single **First targets per Fitts file** input removes the first **N targets per Fitts file for all participants and sessions**, not entire tests or repetitions of each condition. Their start-to-click intervals (including endpoints) are also excluded from head statistics, including initial pose, using shared host timestamps. Missing shared timestamps produce an explicit error. Source recordings and saved calibration remain unchanged.
In **Fitts calculation review**, enable **Add MTnodwell, no-dwell throughput and dwell behaviour**. Only MTnodwell and no-dwell throughput are added to the Fitts tables; all dwell behaviour estimates and counts have a separate **Dwell behaviour stats** table, collapsed session table and CSV. Matched results also separate the two tables/CSVs. Dwell-only fields show **-** for non-dwell/unknown selection, not zero; numeric exports keep these missing values blank. Original MT/throughput remain unchanged; MTnodwell subtracts configured dwell for dwell targets and keeps MT for known non-dwell methods. Unknown methods and negative adjusted timings are counted and excluded only from adjusted timing. Dwell estimates report cumulative inside-target time, first-entry-to-final-residence settling time, final uninterrupted residence, cursor path/ranges and head angular path/ranges during all inside-target visits. These are sample-based estimates, not software timer events; cumulative residence need not exceed configured dwell. Missing paths or shared-clock head coverage yield unavailable estimates and contributing counts; typing geometry is unsupported for spatial estimates. Global target exclusions apply first, matching applies only to the separate comparison, and the two existing opening-duration controls keep their original scopes. See the README for interpolation details and limitations.
Enable **Show matched-target comparison** below Fitts stats. In the collapsed **Fitts calculation review**, the **Matching method** choice offers the original **Strict planned ID (within 0.1 bits)** or **Closest target conditions (retain every test)**. The latter keeps the maximum equal target count per shared block/amplitude/width condition after exclusions, reserves contributions from every Fitts test, balances remaining per-test quotas and chooses nearest available planned IDs without a hard cutoff. It reports per-test counts and the largest ID difference from reference targets; geometry/IDs are never changed. With no exclusions, the supplied data retain 24 targets each (12 per test); with N=2, 20 each (10 per test). Actual planned IDs still differ because the original layouts differ. Selection uses geometry and counts, never clicks or movement times, and is deterministic/greedy rather than globally ID-optimal. Other analyses remain unchanged; an impossible all-test comparison produces an explicit notice.
**Fitts test conditions by file** now has one row per participant/Fitts file, derived only from its Fitts trials: amplitude/width lists, block types, observed targets per condition/repetition, observed repetitions per condition and recorded/analysed trial totals. Targets per repetition (e.g. 3 versus 6) are not repetitions per condition; incomplete files cannot establish the intended settings.
The main Fitts table shows **Planned ID average (bits)** immediately before **Effective ID (IDe, bits)**, averaging the geometry-based IDs of analysed trials. The first item under **Fitts stats** is a collapsed **Fitts test conditions by file** table with saved amplitude/width lists, block types, observed targets per repetition, repetition/trial counts and CSV export. The files do not save the complete original settings panel, so observed counts cannot prove intended settings or run completeness. Circle amplitude is the configured radius (subject to screen fitting), not necessarily the actual movement distance. **Path TRE** counts re-entries after the first entry and averages them across usable paths, including zero-re-entry paths; it can therefore be below 1.
The Fitts difficulty column is **Effective ID (IDe, bits)**, which depends on endpoint accuracy and can differ for the same planned task. Planned **ID = log2(D/W + 1)** uses saved start/target-centre distance D and width W, not clicks or configured circle radius. Neither comparison method forces IDe to match or changes other tables/playback. Expand **Fitts calculation review** for standards caveats: the existing formulas pool conditions/sessions and use Euclidean Ae, unlike sequence-level task-axis ISO-style throughput. Both tables retain those legacy descriptive formulas for comparability.
Excluded chart periods are marked by **grey background bands**, not grey trace lines: all traces keep their original colours. Every table, including individual-session tables and range checks, has a collapsed **Field key** using the Fitts stats heading/key/description format, ordered exactly like its columns from left to right.
The two time-exclusion fields sit next to each other to the right of the target-count input, with each label above its box (wrapping on narrow screens). **Skip opening of head recording (s)** defaults to **1.0 seconds** from each recording's first sample. It changes session movement ranges, speed, acceleration and range checks only; 0 includes the full recording. Samples at the cutoff are retained; speed/acceleration never cross an exclusion or file boundary. The original reference pose, saved calibration, raw playback poses, initial-rest results, movement-exercise results and pointing results are preserved by this time control. Session summary counts/durations still describe the original files. The movement table shows retained sample counts and exports `head_opening_exclude_s`; exhausted recordings have blank movement estimates and an explicit notice. Embedded initial-pose and session-statistics panels use the same exclusions as the tables. Playback retains all frames, targets and graph points, with excluded data greyed out rather than removed. The two time controls keep separate statistical scopes: recording opening for session movement statistics; hold opening for initial-pose statistics. Excluded target intervals apply to both.
Six compact tables each contain one row per participant: **Session summary**, **Initial head positions**, **Range of movement stats**, **Head movement stats**, **Fitts targets and hits**, and **Fitts stats**, with CSV downloads and source-file footnotes. Expand **Individual sessions** below each main table to inspect a separate table of session values, identified by participant and session without filename columns. These session tables start collapsed, use the same analysis settings, and do not add observations to participant exports. Initial positions show recorded yaw, yaw from calibration heading, and saved calibration heading. Wireframes combine calibration heading with initial-rest pitch/roll. Recorded yaw is not necessarily a compass heading; gravity cannot determine yaw. Pitch/roll retain recorded tilt. Without calibration, recorded yaw remains available but yaw from calibration heading and plots are unavailable. Head movement ranges use relative pose and show the largest within-recording range per angle, not a range spanning different reference poses. Full recordings include preparation and the movement exercise; expand Movement range checks for pointing-task-only comparisons and warnings about difficulty separating yaw and roll near +/-90 degrees pitch. No automatic drift removal is applied. The movement-exercise table remains task-only. Display/export use one decimal place; calculations retain precision. 

**Skip opening of initial-rest hold (s)** defaults to **1.0 seconds** in the session section. Initial-position measurements use **only `initial_rest.hold` samples**, never the preceding `initial_rest.move` phase. The exclusion starts at each saved hold-start time (or its first labelled sample if no start time was saved), not at the recording start. Set it to 0 to include the full hold. Initial-rest averages, spread (SD), usable sample counts/durations, summary meshes and offsets from rest update immediately. The applied duration is included as `initial_rest_exclude_s` in initial-position and combined summary CSVs. If a hold has no samples left, its rest-pose estimates and offsets are blank with an explicit notice; saved calibration heading remains available. Saved calibration, playback, directional holds, whole-recording Head movement stats and pointing results are unchanged.

**IMU versus camera references:** xIMU3/AirPods initial pitch and roll are gravity-referenced tilt estimates, subject to sensor fusion and mounting/fit; webcam pitch and roll are camera-referenced, not gravity-referenced. xIMU3 yaw may be magnetic when enabled and reliable; AirPods yaw has an arbitrary heading reference. Expand **Angle references and zero-position diagrams** in the initial-position section for head-mesh diagrams of pitch/roll zero, absolute/calibration yaw zero (provider heading), and relative yaw zero (mean calibration heading). Its example calibration heading is illustrative, not participant data. Movement ranges and playback use neutral-relative Euler components, not gravity-referenced tilt ranges; a tilted neutral can couple heading changes into pitch/roll.

Participant pose plots include signed head axes: X forward, Y left, Z up. Positive yaw turns left, positive pitch looks up, positive roll tilts the left ear down; negative values reverse those directions. **Zero yaw / Zero pitch / Zero roll** controls above the summaries redraw every participant plot with selected angles set to 0.0 degrees; uncheck to restore. These overrides affect plots only, never tables, CSVs or playback. Source-file lists, angle references, calculation notes and the **Field key** under every table start collapsed. Expand each key for field definitions, units, aggregation and Fitts/path formulas. Table columns fit their content and headers are bold.

Target exclusions update both pointing statistics and head statistics for those target intervals. Initial-rest and recording-opening exclusions keep their separate statistical scopes as described above. Raw recordings and playback retain all data, with exclusions shown in grey; the embedded statistics update with the tables. If N exhausts a pointing file, that file contributes zero analysed pointing targets; unavailable measures are blank. Original recorded sample totals/durations remain clearly labelled as recording metadata rather than analysed counts.

The viewer has play/pause, rewind, seek, 0.1–5x speed, graph zoom, a rotatable head mesh, live values, target/click markers and pre-test playback. 
The browser renderer replaces Qt/OpenGL; recording times, neutral pose, coordinate transforms, angular dynamics and summary formulas follow the desktop viewer.

For recordings already on Google Drive, mount Drive in a separate cell and call:
```python
viewer.add_session('P03', 'Test 1',
    head_path='/content/drive/MyDrive/study/head.json',
    fitts_path='/content/drive/MyDrive/study/fitts.json')
```
Only run the viewer cell once per collection: rerunning it starts a new empty collection.